# 第四讲 — 均衡与 Aiyagari 模型 (Equilibrium and Aiyagari)

**异质性主体宏观经济学的计算方法 (Computational Methods for Heterogeneous-Agent Macro)**

孙杰

#### Aiyagari 模型

与许多模型一样,Aiyagari 模型由三个模块组成:
1. 由**阶段 (Stages)** 构成的家庭模块
2. 将资本 (capital) 和劳动 (labour) 转化为产品的厂商部门
3. 均衡 (equilibrium) 的概念(稳态 (steady state))

熟悉的经济学:
1. 需求侧
2. 供给侧
3. 均衡的概念

### 环境

In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using Plots, Random, LinearAlgebra

# 核心代码

## 家庭模块

### 参数

In [ ]:
u(c) = c <= 0 ? -Inf : log(c)
loggrid(lo, hi, N) = exp.(range(log(lo), log(hi); length=N))

In [ ]:
function get_params(;
        β=0.96,
        R=1.04,
        b_grid=loggrid(0.05, 200.0, 400),
        z_grid=[2.7, 2.6, 2.5],
        P_z=[0.95 0.04 0.01;
           0.05 0.9  0.05;
           0.01 0.04 0.95],
    )

    return params = (;β, R,
                    V_shape=(length(b_grid), length(z_grid)),
                    b_grid, z_grid, P_z)
end

### 效用函数和对数网格

In [ ]:
"""
将值 `x` 对齐到最近的网格索引。

"""
snap_idx(grid, x) = argmin(abs.(grid .- x))

### 分阶段的后向迭代 (backward iteration)

三个阶段组合为对二维价值函数 $V \in \mathbb{R}^{N_b \times N_z}$ 的一次后向步骤。每个阶段将一个带标签的 $V$ 映射到下一个:

- **阶段 3 (消费-储蓄, Consumption-Saving) 后向。** 网格搜索 $V^{\mathrm{end}} \mapsto V$。
- **阶段 2 (收入, Income) 后向。** 在 $R\,b^{\mathrm{end}} + z_{\mathrm{grid}}[j]$ 处查表 $V \mapsto V^{\mathrm{start}}_{\mathrm{post}}$。
- **阶段 1 (收入冲击, Income Shock) 后向。** 通过 $\Pi^\top$ 矩阵乘法 $V^{\mathrm{start}}_{\mathrm{post}} \mapsto V^{\mathrm{start}}_{\mathrm{pre}}$。

在期间边界处,$V^{\mathrm{end}} = \beta\, V^{\mathrm{start}}_{\mathrm{pre}}$。

In [ ]:
"""
阶段 1 (收入冲击) 后向: V_start = V_pre_inc * P_z'。

"""
income_shock_backward(V_post_inc_shock, params) = V_post_inc_shock * params.P_z'

"""
阶段 2 (收入) 后向: V_pre_inc(b_end, z) = V(R*b_end + z, z),对齐到最近网格点。
完全广播 —— 无显式循环。

"""
function income_backward(V_post_inc, params)
    (;R, b_grid, z_grid) = params
    b_post_inc = snap_idx.(Ref(b_grid), R .* b_grid .+ z_grid')
    return V_pre_inc = V_post_inc[CartesianIndex.(b_post_inc, axes(b_post_inc, 2)')]
end

"""
阶段 3 (消费-储蓄) 后向: 在 b_end 选择上对 V_end → V 进行网格搜索。

"""
function consumption_saving_backward(V_end, params)
    # 解包 b_grid
    (;b_grid) = params
    
    # 将 b_grid 移到维度 3 以表示 b_end 的选择
    b_next_grid = insertdims(b_grid; dims=(1,2))

    # 将 V_end 的 b_grid 部分移到维度 3 以表示 b_end 的选择
    # V_end_reshape 形状现为 [1, z_grid, b_grid]
    V_end_reshape = insertdims(permutedims(V_end, (2,1)); dims=1)

    # 在 b_end 维度 (dim=3) 上求最大
    V = maximum(u.(b_grid .- b_next_grid) .+ V_end_reshape; dims=3)

    # 从 V 中删除维度 3 并返回
    return dropdims(V; dims=3)
end

"""
通过 argmax 而非 max 从 V_end 中读取最优 b_end 政策。

"""
function policy_function(V_end, params)
    # 解包 b_grid
    (;b_grid) = params
    
    # 将 b_grid 移到维度 3 以表示 b_end 的选择
    b_next_grid = insertdims(b_grid; dims=(1,2))

    # 将 V_end 的 b_grid 部分移到维度 3 以表示 b_end 的选择
    # V_end_reshape 形状现为 [1, z_grid, b_grid]
    V_end_reshape = insertdims(permutedims(V_end, (2,1)); dims=1)

    # 在 b_end 维度 (dim=3) 上求最大
    policy_idxs = argmax(u.(b_grid .- b_next_grid) .+ V_end_reshape; dims=3)

    return [idx[3] for idx in policy_idxs[:,:,1]]
end

In [ ]:
function bellman_operator(V_end, params)
    (;β) = params
    # 阶段 3
    V_post_inc = consumption_saving_backward(V_end, params)

    # 阶段 2
    V_post_inc_shock = income_backward(V_post_inc, params)

    # 阶段 1
    V_start = income_shock_backward(V_post_inc_shock, params)

    # 时间的流逝
    V_end_new = β .* V_start
    return V_end_new
end

### 价值函数迭代 (Value Function Iteration)

迭代 $V \mapsto \mathcal{T}V$ 直到最大绝对变化降至 `tol` 以下。

In [ ]:
function solve_vfi(params; tol=1e-6, maxiter=2000, verbosity=0)
    V = zeros(params.V_shape)

    Δ, iters = Inf, 0
    while Δ >= tol
        TV = bellman_operator(V, params)
        Δ  = maximum(abs.(TV .- V))
        V  = TV
        iters += 1
        iters > maxiter && error("VFI did not converge in $maxiter iterations")
    end

    verbosity >= 1 && println("VFI converged in $iters iterations with error $Δ")
    return V
end

In [ ]:
params = get_params()
V = solve_vfi(params; verbosity=1)

### 分阶段的前向迭代 (forward iteration)

我们可以将每个人口矩阵逐阶段向前模拟。

- **阶段 1 (收入冲击) 前向。** $\lambda \mapsto \lambda\, \Pi$。
- **阶段 2 (收入) 前向。** 每个 $(b^{\mathrm{end}}, z)$ 单元移动到 $(R\,b^{\mathrm{end}} + z_{\mathrm{grid}}[j],\, z)$。
- **阶段 3 (消费-储蓄) 前向。** 每个 $(b, z)$ 单元移动到 $(b - c^\star(b, z),\, z)$。

两次财富重新分箱 (re-bin) 都对齐到最近网格点。

In [ ]:
"""
阶段 1 前向: λ_post = λ_pre * P_z。

"""
income_shock_forward(λ, P_z) = λ * P_z

"""
阶段 2 前向: 每个 (i_b_end, i_z) 单元移动到 (snap(R*b_end + z), i_z)。
通过广播向量化目标索引;一次紧凑的 scatter 操作。

"""

function change_λ_wealth(λ, b_inds_new)
    λ_new = zero(λ)
    for old_idx in CartesianIndices(λ)
        λ_new[b_inds_new[old_idx], old_idx[2]] += λ[old_idx]
    end
    return λ_new
end

function income_forward(λ, params)
    (;R, b_grid, z_grid) = params
    dest = snap_idx.(Ref(b_grid), R .* b_grid .+ z_grid')               # (N_b, N_z)
    return change_λ_wealth(λ, dest)
end

"""
阶段 3 前向: 每个 (i_b, i_z) 单元移动到 (c_ind[i_b, i_z], i_z)。
c_ind 恰好就是目标索引 —— 直接 scatter。

"""
consumption_saving_forward(λ, b_inds_new, params) = change_λ_wealth(λ, b_inds_new)

"""
复合算子 T* 的一次前向步骤:
λ → 收入冲击 → 收入 → 消费储蓄。

"""
function simulate_population_forward(λ, b_inds_new, params)
    # 阶段 1
    λ = income_shock_forward(λ, params.P_z)

    # 阶段 2
    λ = income_forward(λ, params)

    # 阶段 3
    λ = consumption_saving_forward(λ, b_inds_new, params)
    
    return λ
end


In [ ]:
function find_steady_state_population(V_end, params; tol=1e-5, maxiter=10_000, verbosity=0)

    b_inds_new = policy_function(V_end, params)
    
    λ = fill(1/length(params.V_shape), params.V_shape) # 初始将家庭均匀分布在各网格点上

    Δ, iters = Inf, 0
    while Δ >= tol
        λ_new = simulate_population_forward(λ, b_inds_new, params)
        Δ = maximum(abs.(λ_new .- λ))
        λ = λ_new
        iters += 1
        iters > maxiter && error("Steady state λ did not converge in $maxiter iterations")
    end

    verbosity >= 1 && println("Steady state λ converged in $iters iterations with error $Δ")
    return λ
end

In [ ]:
function solve_aiyagari_steady_state(params; verbosity=0)
    V_end = solve_vfi(params; verbosity)
    λ = find_steady_state_population(V_end, params; verbosity)
    return (;V_end, λ)
end

function solve_aiyagari_steady_state(;verbosity=0, param_vals...)
    return solve_aiyagari_steady_state(get_params(;param_vals...); verbosity)
end

In [ ]:
params = get_params(;R=1.04)
(;V_end, λ) = solve_aiyagari_steady_state(params; verbosity=0)

In [ ]:
sum(λ .* params.b_grid) # 经济体中的总资本

In [ ]:
params.b_grid

## 计算矩 (Moments)

### 由 $V$ 和 $c^\star$ 计算的横截面 (cross-sectional) 总量

每期的总福利 $\bar V$、总消费 $\bar C$、总财富 $\bar K$ —— 均为与财富边际分布的内积。

In [ ]:
get_c_policy(b_grid, b_grid_policy) = b_grid .- b_grid[b_grid_policy]

function compute_household_aggregates(;param_vals...)
    params = get_params(;param_vals...)
    (V_end, λ) = solve_aiyagari_steady_state(params)

    
    b_grid_policy = policy_function(V_end, params)
    c_policy = get_c_policy(params.b_grid, b_grid_policy)
    
    L_agg = sum(λ)
    K_agg = sum(params.b_grid .* λ)
    V_agg = sum(V_end .* λ)
    C_agg = sum(c_policy .* λ)

    return (;V_end, λ, L_agg, K_agg, V_agg, C_agg)
end

In [ ]:
params = get_params(;R=1.04)
(V_end, λ) = solve_aiyagari_steady_state(params)

@show V_agg = sum(V_end .* λ)
@show K_agg = sum(params.b_grid .* λ)

b_grid_policy = policy_function(V_end, params)
c_grid_policy = get_c_policy(params.b_grid, b_grid_policy)

@show C_agg = sum(c_grid_policy .* λ)

@show L_agg = sum(λ)


In [ ]:
compute_Y(K, L; α=1/3) = K^α * L^(1-α)

In [ ]:
res = compute_household_aggregates(;R=1.03)

In [ ]:
Y_agg = compute_Y(res.K_agg, res.L_agg)

In [ ]:
res.C_agg

In [ ]:
function compute_excess_demand(R)
    res = compute_household_aggregates(;R)
    
    Y_agg = compute_Y(res.K_agg, res.L_agg)

    return excess_demand = (res.C_agg - Y_agg)/Y_agg
end

In [ ]:
compute_excess_demand(1.05)

In [ ]:
tol = 0.01
lr = 0.001

R = 1.01
err = Inf
while err > tol
    excess_demand = compute_excess_demand(R)
    
    R_new = R - excess_demand*lr
    err = abs(excess_demand)

    R = R_new
    
    println("R=$R\t excess_demand=$excess_demand")
end

println("Equilibrium R: $R")